In [1]:
!pip install -q tqdm
!pip install -q ipdb
!pip install -q sentencepiece
!pip install -q wandb
!pip install requests

#Libraries
import os, sys
import ipdb
from tqdm import tqdm
from datetime import datetime
import platform, shutil
import requests, zipfile, io

#Pytorch
import torch
import torch.nn as nn
from torch.nn import functional as F

#Tokenizer
import sentencepiece as spm

#Architeture
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

#GPU Cache Memory
torch.cuda.empty_cache()


#Architecture Parameters
batch_size = 32
context = 512
embed_size = 384
n_layers = 7
n_heads = 7
BIAS = True

#Hyperparameters
lr = 3e-4
dropout = 0.05
weight_decay = 0.01
grad_clip = 1.0

#Training parameters
train_iters = 100000
eval_interval = 50
eval_iters = 10
compile = False
checkpoint_dir = 'models'
checkpoint_fn = 'latest.pt'
checkpoint_load_fn = 'latest.pt'
dtype = torch.bfloat16

#MODE
inference = False

#DEVICE
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device You will be using: ", device)

device You will be using:  cpu


In [3]:
wandb_log = True
wandb_project = "llm1"
wandb_run_name = "llm1-"+datetime.now().strftime("%Y_%m_%d_%H_%M_%S")

if wandb_log:
  import wandb
  wandb.init(project=wandb_project, name=wandb_run_name)

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: apolloolympos287 (apolloolympos287-adobe) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
import os
import torch

class BPETokenizer:
    def __init__(self):
        self.vocab_size = 50257
        chars = sorted(list(set(" abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ.,!'-_0123456789\n")))
        self.stoi = {ch: i for i, ch in enumerate(chars)}
        self.itos = {i: ch for i, ch in enumerate(chars)}

    def get_piece_size(self):
        return self.vocab_size
    def Encode(self, text):
        return [self.stoi.get(c, 32) for c in text]
    def Decode(self, ids):
        return "".join([self.itos.get(i, " ") for i in ids])

sp = BPETokenizer()
vocab_size = sp.get_piece_size()
encode = lambda s: sp.Encode(s)
decode = lambda l: sp.Decode(l)

if os.path.exists("encoded_data.pt"):
    data = torch.load('encoded_data.pt')
else:
    text = "Once upon a time there was a small language model learning how to speak using PyTorch. " * 100
    data = torch.tensor(encode(text), dtype=torch.long)

data_size = len(data)
spl = int(0.9 * data_size)
train_data = data[:spl]
val_data = data[spl:]

In [5]:
encode = lambda s: sp.Encode(s)
decode = lambda l: sp.Decode(l)

print(encode("Once upon a time"))
print(decode(encode("Once upon a time")))

[31, 57, 46, 48, 1, 64, 59, 58, 57, 1, 44, 1, 63, 52, 56, 48]
Once upon a time


In [6]:
if os.path.exists(f"encoded_data.pt"):
  data = torch.load('encoded_data.pt')
else:
  data = torch.tensor(encode(text), dtype=torch.long)
  torch.save(data, 'encoded_data.pt')

In [7]:
data_size=len(data)
spl = int(0.9*data_size)
train_data=data[:spl]
val_data=data[spl:]

print(f"Total data: {data_size/1e6:2f} Million | Training: {len(train_data)/1e6:2f} Million | Validation: {len(val_data)//1e6:2f}")

Total data: 0.008700 Million | Training: 0.007830 Million | Validation: 0.000000


In [8]:
import torch

data = torch.randint(0, 50257, (1000000,), dtype=torch.long)

data_size = len(data)
spl = int(0.9 * data_size)
train_data = data[:spl]
val_data = data[spl:]


In [9]:
def get_batch(split):
  #BS = Batch Size
  data = train_data if split=="train" else val_data
  inds = torch.randint(len(data) - context, (batch_size,))
  x = torch.stack ([data[i: i + context]  for i in inds]) # (BS, SL)
  y = torch.stack ([data[i+1: i+context+1] for i in inds]) # (BS, SL)

  x,y = x.to(device), y.to(device)
  return x,y

x,y = get_batch("train")
print(x.shape, y.shape)
print(x[0][:10])
print(y[0][:10])

torch.Size([32, 512]) torch.Size([32, 512])
tensor([ 1013, 40843, 47455,  3577,   535,  2954,  1861, 49025, 34763,  8864])
tensor([40843, 47455,  3577,   535,  2954,  1861, 49025, 34763,  8864,  3882])


In [10]:
class GPT(nn.Module):
  def __init__(self):
    super().__init__()
    self.embeddings =  nn.Embedding(vocab_size, embed_size)
    self.positions = nn.Embedding(context, embed_size)
    self.blocks = nn.Sequential( *[Block (n_heads) for _ in range(n_layers)])
    self.ln = nn.LayerNorm(embed_size)
    self.final_linear = nn.Linear(embed_size, vocab_size, bias=BIAS)

  def _init_weights(self, module):
    if isinstance(module, nn.Linear):
      torch.nn.init.normal_(module.weight, mean = 0.0, std = 0.02)
      if module.bias is not None:
          torch.nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
        torch.nn.init.normal_(module.weight, mean = 0.0, std = 0.02)

  def forward(self, input, targets=None):

    loss = None
    BS,SL = input.shape
    emb = self.embeddings(input)
    pos = self.positions(torch.arange(SL, device=device))
    x = emb + pos
    x = self.blocks(x)
    x = self.ln(x)
    logits = self.final_linear(x)

    if targets is not None:
      BS, SL, VS = logits.shape
      logits = logits.view(BS*SL, VS)
      targets = targets.view(BS*SL)
      loss = F.cross_entropy(logits, targets)

      #Manual calculation
      counts = logits.exp()
      prob = counts / counts.sum (-1, keepdim = True)
      loss2 = - prob[torch.arange(BS*SL), targets].log().mean()

      # cross entropy = - Log p(x)

      #if(not torch.allclose(loss, loss2)):
      #  print(f"[Loss Diff] Pytorch:{loss.item()} Manual:{loss2.item()}")

    return logits, loss

  def generate(self, input, max=500):
    for _ in range (max):
      input = input[:,-context:]
      logits, _ = self (input)
      logits = logits[:, -1,:]
      probs = F.softmax (logits, dim=-1)
      next = torch.multinomial(probs, num_samples=1)
      input = torch.cat((input, next), dim=1)
    return input

In [11]:
class Block(nn.Module):
  def __init__(self, n_heads):
    super().__init__()
    head_size = embed_size // n_heads
    self.ma = Multihead (n_heads, head_size)
    self.feed_forward = ForwardLayer (embed_size)
    self.ln1 = nn.LayerNorm(embed_size)
    self.ln2 = nn.LayerNorm(embed_size)

  def forward(self, x):
    x = x + self.ma(self.ln1(x))
    x = x + self.feed_forward(self.ln2(x))
    return x

In [12]:
class ForwardLayer(nn.Module):
  def __init__(self, embed_size):
    super().__init__()
    self.network = nn.Sequential(
        nn.Linear(embed_size, 6*embed_size, bias = BIAS),
        nn.GELU(),
        nn.Linear(6*embed_size, embed_size, bias = BIAS),
        nn.Dropout(dropout)
    )
  def forward(self, x):
    x = self.network(x)
    return x

In [13]:
class Multihead(nn.Module):
  def __init__(self, n_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range (n_heads)])
    self.combine = nn.Linear(head_size * n_heads, embed_size, bias=BIAS)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    x = torch.cat([head(x) for head in self.heads], dim=-1)
    x = self.combine(x)
    x = self.dropout(x)
    return x

In [14]:
class Head(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    self.queries= nn.Linear(embed_size, head_size, bias=BIAS)
    self.keys= nn.Linear(embed_size, head_size, bias=BIAS)
    self.values= nn.Linear(embed_size, head_size, bias=BIAS)

    self.register_buffer('tril', torch.tril(torch.ones(context, context)))
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    BS, SL, VS = x.shape
    q=self.queries(x)
    k=self.keys(x)
    v=self.values(x)

    attn_w = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5
    attn_w = attn_w.masked_fill(self.tril[:SL,:SL]==0, float('-inf'))
    attn_w = F.softmax(attn_w, dim = -1)

    x = attn_w @ v

    return x


In [15]:
torch.tril(torch.ones(context, context))

tensor([[1., 0., 0.,  ..., 0., 0., 0.],
        [1., 1., 0.,  ..., 0., 0., 0.],
        [1., 1., 1.,  ..., 0., 0., 0.],
        ...,
        [1., 1., 1.,  ..., 1., 0., 0.],
        [1., 1., 1.,  ..., 1., 1., 0.],
        [1., 1., 1.,  ..., 1., 1., 1.]])

In [16]:
head_size = embed_size // n_heads
print(f"embed: {embed_size} n_heads: {n_heads} head_size {head_size}")

embed: 384 n_heads: 7 head_size 54


In [17]:
x,y = get_batch("train")
print(x.shape, y.shape)
#print(x[0][:10])
#print(y[0][:10])

model = GPT()
model = model.to(dtype)
model = model.to(device)

embeddings = nn.Embedding(vocab_size, embed_size).to(device)
positions = nn.Embedding(context, embed_size).to(device)
queries = nn.Linear(embed_size, head_size, bias=BIAS).to(device)
keys = nn.Linear(embed_size, head_size, bias=BIAS).to(device)
values = nn.Linear(embed_size, head_size, bias=BIAS).to(device)
tril=torch.tril(torch.ones(context,context)).to(device)

emb = embeddings(x)
pos = positions(torch.arange(context, device=device))
x = emb + pos

q=queries(x)
k=keys(x)
v=values(x)
print(q.shape, k.shape, v.shape)
torch.set_printoptions(precision=2, sci_mode=False)
print(q[0][0][:5])
#torch.set_printoptions(precision=4, threshold=1000, edgeitems=3, linewidth=80, profile='default', sci_mode=True)

attn_w = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5
attn_w = attn_w.masked_fill(tril[:context,:context]==0, float('-inf'))
attn_w = F.softmax(attn_w, dim=-1)
x = attn_w @ v


torch.Size([32, 512]) torch.Size([32, 512])
torch.Size([32, 512, 54]) torch.Size([32, 512, 54]) torch.Size([32, 512, 54])
tensor([ 0.80,  1.39, -0.40,  0.23,  1.08], grad_fn=<SliceBackward0>)


In [18]:
# Understand Attention Matrix
full = q @ k.transpose(-2, -1)

a = q[0][5]
b = k.transpose(-2, -1)[0, :, 5]
print(a, b)
c = torch.dot(a, b)
print(c)
print(full[0][5][5])


tensor([-0.52,  1.69, -0.15, -0.14,  0.56,  0.30,  0.11,  0.80, -0.25,  1.09,
        -1.22,  0.48,  2.27, -2.12, -0.17,  1.16, -0.15,  0.52, -0.43, -0.24,
        -0.56, -0.56, -0.09, -0.00, -0.14, -0.59, -1.13,  0.63, -0.29,  0.09,
        -0.99, -0.30, -0.25,  0.15, -1.07, -1.64,  1.16,  0.01, -0.67, -0.48,
        -0.23, -1.58, -0.22, -0.11, -0.35,  0.91,  0.34,  0.93, -0.15,  0.07,
        -0.63,  0.10, -0.58,  0.31], grad_fn=<SelectBackward0>) tensor([-1.20, -0.17, -0.05, -0.26, -0.11, -0.23,  0.94, -0.42,  0.62, -0.55,
         0.41, -0.16, -0.18,  1.05,  0.42, -1.34, -1.16, -1.03,  0.29,  0.14,
         0.02, -1.26,  0.59,  0.06, -0.73,  1.24, -0.82, -0.57, -0.48, -0.25,
         0.19,  1.36,  0.14, -0.14,  1.77, -0.06,  0.37,  1.47,  0.75,  0.56,
        -0.69,  0.21,  0.24,  0.16,  2.52,  0.45, -0.00,  0.40, -0.96, -0.23,
         0.54, -1.22, -0.39, -1.53], grad_fn=<SelectBackward0>)
tensor(-9.11, grad_fn=<DotBackward0>)
tensor(-9.11, grad_fn=<SelectBackward0>)


In [19]:
# Understand the updating of the V content
print(attn_w.shape, v.shape)

print(x[0][7])
# x = attn_w @ v  # 512 x 54

attn_scores2 = attn_w[0, 7, :]  # Shape [512]
# Initialize a tensor to store the result
result = torch.zeros(54)
# Compute the dot product for each column in v for the first token in the first batch
for i in range(54):
     result[i] = torch.dot(attn_scores2, v[0, :, i])
print(result)


torch.Size([32, 512, 512]) torch.Size([32, 512, 54])
tensor([-0.60,  0.48, -0.12, -0.57, -0.13, -0.08, -0.18, -0.42,  0.39,  0.09,
         0.04,  0.10, -0.53,  0.27, -0.03,  0.14, -0.60, -0.23, -0.48, -0.33,
        -0.17,  0.61,  0.39, -0.37,  0.23,  0.37,  0.02, -0.03,  0.14,  0.17,
         0.40, -0.17,  0.21,  0.31,  0.53,  0.36,  0.48, -0.27, -0.03, -0.14,
        -0.25, -0.12, -0.58, -0.13,  0.10,  0.20,  0.70,  0.40, -0.04, -0.39,
        -0.01, -0.28, -0.19,  0.09], grad_fn=<SelectBackward0>)
tensor([-0.60,  0.48, -0.12, -0.57, -0.13, -0.08, -0.18, -0.42,  0.39,  0.09,
         0.04,  0.10, -0.53,  0.27, -0.03,  0.14, -0.60, -0.23, -0.48, -0.33,
        -0.17,  0.61,  0.39, -0.37,  0.23,  0.37,  0.02, -0.03,  0.14,  0.17,
         0.40, -0.17,  0.21,  0.31,  0.53,  0.36,  0.48, -0.27, -0.03, -0.14,
        -0.25, -0.12, -0.58, -0.13,  0.10,  0.20,  0.70,  0.40, -0.04, -0.39,
        -0.01, -0.28, -0.19,  0.09], grad_fn=<CopySlices>)


In [20]:
@torch.no_grad()
def generate_sample(input):
  t1 = torch.tensor(encode(input), dtype = torch.long, device = device)
  t1 = t1[None, :]
  newgen = model.generate(t1, max=64)[0].tolist()
  result=decode(newgen)
  print(f"{result}")

generate_sample("Once upon a time")

Once upon a time                                                                
